# 00 — Preparación y validación

Este notebook crea la entrada analítica común del dashboard mediante una **transformación mínima** del archivo procesado existente `indice_localidad_disposicion_inadecuada.csv`.

No reconstruye la encuesta, los delitos, la población ni el indicador de disposición. Solo selecciona las variables existentes, normaliza los nombres visibles de las localidades, valida tipos y escalas, y guarda `dashboard_dataset.csv` porque la especificación exige una entrada tabular común para los notebooks 01–07.

## 1. Configuración de rutas e importaciones

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "processed").exists():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del repositorio o desde notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"


In [2]:
import pandas as pd
from src.dashboard_utils import (
    DASHBOARD_DATASET,
    NUMERIC_COLUMNS,
    REQUIRED_COLUMNS,
    SOURCE_DATASET,
    prepare_dashboard_dataset,
    validate_dataset,
)

source_path = PROCESSED_DIR / SOURCE_DATASET
assert source_path.exists(), f"No existe el insumo procesado: {source_path}"
print(f"Insumo reutilizado: {source_path.relative_to(PROJECT_ROOT)}")

Insumo reutilizado: data/processed/indice_localidad_disposicion_inadecuada.csv


## 2. Consolidación mínima

Las estadísticas y la metodología se conservan tal como están en el dataset procesado. Los porcentajes ya se encuentran en escala 0–100.

In [3]:
df, validation_report, output_path = prepare_dashboard_dataset(PROJECT_ROOT)
print(f"Archivo analítico generado: {output_path.relative_to(PROJECT_ROOT)}")
validation_report

Archivo analítico generado: data/processed/dashboard_dataset.csv


{'filas': 19,
 'localidades': 19,
 'duplicados': 0,
 'nulos': 0,
 'escala_porcentajes': '0–100'}

## 3. Validaciones requeridas

In [4]:
df.shape

(19, 10)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   locality                    19 non-null     str    
 1   official_crimes_2025        19 non-null     float64
 2   estimated_crimes_2025       19 non-null     float64
 3   population_2025             19 non-null     float64
 4   crime_rate_100k             19 non-null     float64
 5   insecurity_noche_pct        19 non-null     float64
 6   crime_z                     19 non-null     float64
 7   insecurity_z                19 non-null     float64
 8   perception_excess_index     19 non-null     float64
 9   disposicion_inadecuada_pct  19 non-null     float64
dtypes: float64(9), str(1)
memory usage: 1.6 KB


In [6]:
df.isna().sum()

locality                      0
official_crimes_2025          0
estimated_crimes_2025         0
population_2025               0
crime_rate_100k               0
insecurity_noche_pct          0
crime_z                       0
insecurity_z                  0
perception_excess_index       0
disposicion_inadecuada_pct    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.describe()

,official_crimes_2025,estimated_crimes_2025,population_2025,crime_rate_100k,insecurity_noche_pct,crime_z,insecurity_z,perception_excess_index,disposicion_inadecuada_pct
count,19.000000,19.000000,1.900000e+01,19.000000,19.000000,19.000000,1.900000e+01,19.000000,19.000000
mean,8601.894737,21342.873819,4.175921e+05,8090.018265,43.022857,0.215310,4.031863e-16,-0.215310,21.409821
std,4550.968589,13896.648990,3.563676e+05,7582.410812,7.409333,0.363916,1.027402e+00,1.118147,9.558611
min,1758.000000,1696.249945,1.680100e+04,2247.860395,28.593890,-0.272754,-2.000768e+00,-2.278909,5.708588
25%,5902.500000,9094.860099,1.486610e+05,3713.496058,37.867881,-0.024660,-7.148058e-01,-1.104561,16.289393
50%,8981.000000,18766.713245,3.791090e+05,5695.553251,41.526423,0.187033,-2.075004e-01,-0.143813,22.751170
75%,9761.000000,31385.334020,6.265655e+05,7273.782600,48.299861,0.308012,7.317267e-01,0.611880,25.662036
max,17545.000000,45142.597989,1.246637e+06,28012.846671,59.164545,0.975000,2.238259e+00,1.476154,42.264985


In [9]:
locality_count = df["locality"].nunique()
assert locality_count == len(df) == 19
assert df["disposicion_inadecuada_pct"].between(0, 100).all()
assert df["insecurity_noche_pct"].between(0, 100).all()
assert set(NUMERIC_COLUMNS).issubset(df.select_dtypes(include="number").columns)
assert output_path.exists() and output_path.stat().st_size > 0
print(f"Validación aprobada: {locality_count} localidades, sin duplicados ni nulos; porcentajes en escala 0–100.")

Validación aprobada: 19 localidades, sin duplicados ni nulos; porcentajes en escala 0–100.


## 4. Trazabilidad

- Entrada: `data/processed/indice_localidad_disposicion_inadecuada.csv`.
- Salida: `data/processed/dashboard_dataset.csv`.
- Transformaciones: selección de las diez variables analíticas existentes, normalización ortográfica de localidades, ordenamiento y validación.
- No se modificó ningún archivo procesado previo.